# Spaceship Titanic — Kaggle Competition
**Goal:** Predict which passengers were transported to an alternate dimension.

**Metric:** Classification Accuracy

**Pipeline:**
1. EDA
2. Data Cleaning & Feature Engineering
3. Encoding
4. Modeling (Logistic Regression → Random Forest → LightGBM)
5. Submission

## 1. Imports & Load Data

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

print('Train shape:', train.shape)
print('Test shape:', test.shape)

Train shape: (8693, 14)
Test shape: (4277, 13)


## 2. EDA

In [2]:
train.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [3]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


In [4]:
# Missing values
print(train.isna().sum())

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64


In [5]:
# Target distribution — check if classes are balanced
print(train['Transported'].value_counts())

Transported
True     4378
False    4315
Name: count, dtype: int64


In [6]:
# Transportation rate by HomePlanet
print(train.groupby('HomePlanet')['Transported'].mean())

HomePlanet
Earth     0.423946
Europa    0.658846
Mars      0.523024
Name: Transported, dtype: float64


In [7]:
# CryoSleep vs spending — CryoSleep passengers spend nothing
services = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
train['totalSpend_eda'] = train[services].sum(axis=1)
print(train.groupby('CryoSleep')['totalSpend_eda'].mean())
train.drop('totalSpend_eda', axis=1, inplace=True)

CryoSleep
False    2248.299687
True        0.000000
Name: totalSpend_eda, dtype: float64


## 3. Data Cleaning & Feature Engineering

In [8]:
# Split Cabin into Deck, RoomNum, Side
train[['Deck', 'RoomNum', 'Side']] = train['Cabin'].str.split('/', expand=True)
test[['Deck', 'RoomNum', 'Side']] = test['Cabin'].str.split('/', expand=True)

# Split PassengerId into GroupId and PersonId
train[['groupId', 'personId']] = train['PassengerId'].str.split('_', expand=True)
test[['groupId', 'personId']] = test['PassengerId'].str.split('_', expand=True)

# Group size — computed from train only to avoid leakage
group_counts = train['groupId'].value_counts()
train['groupSize'] = train['groupId'].map(group_counts)
test['groupSize'] = test['groupId'].map(group_counts)

# isAlone feature — solo travelers may behave differently
train['isAlone'] = (train['groupSize'] == 1).astype(int)
test['isAlone'] = (test['groupSize'] == 1).astype(int)

# Save PassengerId before dropping
test_passenger_ids = test['PassengerId'].copy()

# Drop columns we don't need
train = train.drop(['Cabin', 'Name', 'PassengerId', 'groupId', 'personId'], axis=1)
test = test.drop(['Cabin', 'Name', 'PassengerId', 'groupId', 'personId'], axis=1)

In [9]:
# Smart NA filling:
# CryoSleep passengers can't spend money — fill their spending with 0
spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
for col in spend_cols:
    train.loc[train['CryoSleep'] == True, col] = train.loc[train['CryoSleep'] == True, col].fillna(0)
    test.loc[test['CryoSleep'] == True, col] = test.loc[test['CryoSleep'] == True, col].fillna(0)

# Fill remaining NAs — median for numerical, mode for categorical (fit on train only)
fill_cols = [col for col in train.columns if col != 'Transported']
for col in fill_cols:
    if train[col].dtype in ['int64', 'float64']:
        median_val = train[col].median()
        train[col] = train[col].fillna(median_val)
        test[col] = test[col].fillna(median_val)
    else:
        mode_val = train[col].mode()[0]
        train[col] = train[col].fillna(mode_val)
        test[col] = test[col].fillna(mode_val)

print('NAs in train:', train.isna().sum().sum())
print('NAs in test:', test.isna().sum().sum())

NAs in train: 0
NAs in test: 0


C:\Users\khale\AppData\Local\Temp\ipykernel_30604\790969287.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train[col] = train[col].fillna(mode_val)
C:\Users\khale\AppData\Local\Temp\ipykernel_30604\790969287.py:18: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  test[col] = test[col].fillna(mode_val)
C:\Users\khale\AppData\Local\Temp\ipykernel_30604\790969287.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to th

In [10]:
# Feature engineering — keep individual spending AND total
train['totalSpend'] = train[spend_cols].sum(axis=1)
test['totalSpend'] = test[spend_cols].sum(axis=1)

## 4. Encoding

In [11]:
# Boolean and string-number columns to int
train[['RoomNum', 'CryoSleep', 'VIP']] = train[['RoomNum', 'CryoSleep', 'VIP']].astype(int)
test[['RoomNum', 'CryoSleep', 'VIP']] = test[['RoomNum', 'CryoSleep', 'VIP']].astype(int)

# One-hot encode categorical columns
train = pd.get_dummies(train, columns=['Side', 'Deck', 'Destination', 'HomePlanet'])
test = pd.get_dummies(test, columns=['Side', 'Deck', 'Destination', 'HomePlanet'])

# Fix whitespace in column names
train.columns = train.columns.str.replace(' ', '_')
test.columns = test.columns.str.replace(' ', '_')

# Align train and test columns
train, test = train.align(test, join='left', axis=1, fill_value=0)

print('Train shape:', train.shape)
print('Test shape:', test.shape)
print('Columns:', train.columns.tolist())

Train shape: (8693, 29)
Test shape: (4277, 29)
Columns: ['CryoSleep', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'Transported', 'RoomNum', 'groupSize', 'isAlone', 'totalSpend', 'Side_P', 'Side_S', 'Deck_A', 'Deck_B', 'Deck_C', 'Deck_D', 'Deck_E', 'Deck_F', 'Deck_G', 'Deck_T', 'Destination_55_Cancri_e', 'Destination_PSO_J318.5-22', 'Destination_TRAPPIST-1e', 'HomePlanet_Earth', 'HomePlanet_Europa', 'HomePlanet_Mars']


## 5. Modeling

In [12]:
X = train.drop('Transported', axis=1)
y = train['Transported']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [13]:
# Baseline: Logistic Regression
numerical_cols = ['Age', 'RoomNum', 'groupSize', 'totalSpend'] + spend_cols
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test_scaled[numerical_cols] = scaler.transform(X_test[numerical_cols])

lr = LogisticRegression(random_state=42)
lr.fit(X_train_scaled, y_train)
print('Logistic Regression:')
print('  Train accuracy:', accuracy_score(y_train, lr.predict(X_train_scaled)))
print('  Test accuracy:', accuracy_score(y_test, lr.predict(X_test_scaled)))

Logistic Regression:
  Train accuracy: 0.7950819672131147
  Test accuracy: 0.7872340425531915


In [14]:
# LightGBM — best performing model
lgbm = lgb.LGBMClassifier(
    learning_rate=0.05,
    max_depth=8,
    n_estimators=100,
    num_leaves=31,
    random_state=42
)
lgbm.fit(X_train, y_train)
print('LightGBM:')
print('  Train accuracy:', accuracy_score(y_train, lgbm.predict(X_train)))
print('  Test accuracy:', accuracy_score(y_test, lgbm.predict(X_test)))

[LightGBM] [Info] Number of positive: 3500, number of negative: 3454
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002240 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1908
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 27
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503307 -> initscore=0.013230
[LightGBM] [Info] Start training from score 0.013230
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
LightGBM:
  Train accuracy: 0.857779695139488
  Test accuracy: 0.8067855089131685


In [15]:
# Cross-validation — more reliable performance estimate
cv_scores = cross_val_score(lgbm, X, y, cv=5, scoring='accuracy')
print('CV scores:', cv_scores)
print('Mean CV accuracy:', cv_scores.mean())

[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001521 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1908
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 27
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380
[LightGBM] [Info] Start training from score 0.014380
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Number of positive: 3502, number of negative: 3452
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000502 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1905
[LightGBM] [Info] Number of data points in the train set: 6954, number of used features: 27
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503595 -> initscore=0.014380


## 6. Submission

In [16]:
# Train final model on all data
final_model = lgb.LGBMClassifier(
    learning_rate=0.05,
    max_depth=8,
    n_estimators=100,
    num_leaves=31,
    random_state=42
)
final_model.fit(X, y)

# Predict on test set
test_features = test.drop('Transported', axis=1)
final_predictions = final_model.predict(test_features)

# Create submission file
submission = pd.DataFrame({
    'PassengerId': test_passenger_ids,
    'Transported': final_predictions
})
submission.to_csv('submission.csv', index=False)
print('Submission saved!')
print(submission.head())

[LightGBM] [Info] Number of positive: 4378, number of negative: 4315
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000762 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1909
[LightGBM] [Info] Number of data points in the train set: 8693, number of used features: 27
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503624 -> initscore=0.014495
[LightGBM] [Info] Start training from score 0.014495
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Submission saved!
  PassengerId  Transported
0     0013_01         True
1     0018_01        False
2     0019_01         True
3     0021_01         True
4     0023_01         True
